<a href="https://colab.research.google.com/github/Anushka-2906/BEE/blob/main/Q4_Writing_Viterbi_Algorithm_for_the_Primer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q4. Viterbi Algorithm (Primer)

In [ ]:
import numpy as np
import math

# Defining HMM Parameters:
states = ['E', '5', 'I']
nucleotides = ['A', 'C', 'G', 'T']

# 2. Initial Probabilities: Probability of starting with an E or an I
initial_probabilities = {'E': 1.0, '5': 0.0, 'I': 0.0}

# 3. Transition matrix: Probabilities of moving from one state to another
transition_probabilities = {
    'Start': {'E': 1.0, '5': 0.0, 'I': 0.0, 'End': 0.0},
    'E': {'E': 0.9, '5': 0.1, 'I': 0.0, 'End': 0.0},
    '5': {'E': 0.0, '5': 0.0, 'I': 1.0, 'End': 0.0},
    'I': {'E': 0.0, '5': 0.0, 'I': 0.9, 'End': 0.1}
}

# 4. Emission probabilities: Probability of emitting nucleotides (A, C, G, T) in each state
emission_probs = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.00, 'G': 0.95, 'T': 0.00},
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4},
}

# Log function to avoid taking log of zero
def log(x):
    if x == 0:
        return -math.inf
    else:
        return math.log(x)

# Log probability of given path
def get_log_prob_of_a_given_path(state_path, observed_sequence):
    log_prob = 0.0
    if len(state_path) != len(observed_sequence):
        raise ValueError("The length of state path and the observed sequence must be the same")

    prev_state = 'Start'
    for i in range(len(observed_sequence)):
        current_state = state_path[i]
        observed_state = observed_sequence[i]
        log_prob += log(transition_probabilities[prev_state][current_state]) + \
                    log(emission_probs[current_state][observed_state])
        prev_state = current_state
    if prev_state == 'I':
        log_prob += log(transition_probabilities[prev_state]['End'])

    return log_prob

# Example State Path and Observed Sequence
state_path = "EEEEEEEEEEEEEEEEEE5IIIIIII"
observed_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"
ans = get_log_prob_of_a_given_path(state_path, observed_sequence)
print(f"Log probability of the given state path: {ans}")

# Viterbi Algorithm Implementation
def viterbiAlgo(observed_sequence):
    num_states = len(states)
    num_observations = len(observed_sequence)
    viterbi_matrix = np.full((num_states, num_observations), -np.inf)
    backpointers = np.zeros((num_states, num_observations), dtype=int)

    # Convert to state indices
    state_to_index = {s: i for i, s in enumerate(states)}

    # Initializing the DP table
    for i, state in enumerate(states):
        viterbi_matrix[i, 0] = log(initial_probabilities[state]) + \
                               log(emission_probs[state][observed_sequence[0]])

    # Recursive part
    for k in range(1, num_observations):
        for curr_idx, current_state in enumerate(states):
            max_log_prob = -np.inf
            best_prev_idx = 0
            for prev_idx, prev_state in enumerate(states):
                log_prob = viterbi_matrix[prev_idx, k - 1] + \
                           log(transition_probabilities[prev_state][current_state])
                if log_prob > max_log_prob:
                    max_log_prob = log_prob
                    best_prev_idx = prev_idx
            viterbi_matrix[curr_idx, k] = max_log_prob + \
                                          log(emission_probs[current_state][observed_sequence[k]])
            backpointers[curr_idx, k] = best_prev_idx

    # Retracing the best path
    best_path = []
    best_final_idx = np.argmax(viterbi_matrix[:, -1])
    best_path.append(states[best_final_idx])

    for i in range(num_observations - 1, 0, -1):
        best_final_idx = backpointers[best_final_idx, i]
        best_path.insert(0, states[best_final_idx])

    return best_path, np.max(viterbi_matrix[:, -1])

# Run Viterbi Algorithm on observed sequence
best_path, log_prob = viterbiAlgo(observed_sequence)
print(f"Most probable path is {best_path} with probability {log_prob}")


Log probability of the given state path: -41.21967768602254
Most probable path is ['E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E'] with probability -38.677666280562796
